In [ ]:
import os
import json
import glob
from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

# Load environment variables
load_dotenv()

In [ ]:
data_dir = "../data"

input_json_dir = os.path.join(data_dir, "2_enrich", "c_attachment_information")

In [ ]:
# Initialize MongoDB client with environment variables
uri = os.getenv("MONGODB_URI")
database_name = os.getenv("MONGODB_DATABASE", "dopegpt")
collection_name = os.getenv("MONGODB_COLLECTION", "messages")

if not uri:
    raise ValueError("MONGODB_URI environment variable is not set")

client = MongoClient(uri, server_api=ServerApi('1'))
client.admin.command('ping')
print("Pinged your deployment. You successfully connected to MongoDB!")

db = client[database_name]
collection = db[collection_name]

In [ ]:
json_files = glob.glob(os.path.join(input_json_dir, "*.json"))
for idx, file in enumerate(json_files, 1):   
    filename = os.path.basename(file)
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    result = collection.insert_one(data)
    document_id = str(result.inserted_id)

    print(f"({idx}/ {len(json_files)}) Uploaded: {filename}")